In [1]:
%matplotlib widget
import csv
import os
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt
from dataclasses import dataclass

# ============================================================
# 1. STYLE & OBJECT CONFIGURATION
# ============================================================
@dataclass
class Spectrum:
    N: str                  # Nombre o Label original
    X: np.ndarray           # X data
    Y: np.ndarray           # Y data
    color: str = None       # Color para ploteo
    linestyle: str = None   # Estilo de línea

    def copy(self):
        """Devuelve una copia profunda del espectro conservando sus propiedades"""
        return Spectrum(
            N=self.N, 
            X=self.X.copy(), 
            Y=self.Y.copy(), 
            color=self.color,         
            linestyle=self.linestyle  
        )

poster_style = {
    "figsize": (8, 5),          
    "dpi": 150,                 
    "font_size": 11,
    "title_size": 14,
    "label_size": 12,
    "legend_size": 10,
    "line_width": 2,
    "font_family": "Arial", # O "Calibri" si lo prefieres
    "grid": True,               
    "colors": ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#56B4E9", "#D55E00", "#F0E442"], 
    "linestyles": ["-", "--", "-.", ":"]
}

# ============================================================
# 2. CORE PLOTTING FUNCTION
# ============================================================
def PlotSpectra(spectra_list, style=poster_style, xlabel="X", ylabel="Y", title="Selected Spectra", 
                xmin=None, xmax=None, ymin=None, ymax=None, save_path=None):
    
    fig, ax = plt.subplots(figsize=style["figsize"], dpi=style["dpi"])
    plt.rcParams.update({"font.size": style["font_size"], "font.family": style["font_family"]})
    
    default_colors = style.get("colors") or ["#000000"]
    default_linestyles = style.get("linestyles") or ["-"]
    
    for i, s in enumerate(spectra_list):
        # Prioridad: Atributo del objeto -> Default del estilo
        c = getattr(s, 'color', None) or default_colors[i % len(default_colors)]
        ls = getattr(s, 'linestyle', None) or default_linestyles[i % len(default_linestyles)]
        
        ax.plot(s.X, s.Y, label=s.N, linewidth=style["line_width"], color=c, linestyle=ls)

    ax.set_xlabel(xlabel, fontsize=style["label_size"])
    ax.set_ylabel(ylabel, fontsize=style["label_size"])
    ax.set_title(title, fontsize=style["title_size"])
    
    if xmin is not None or xmax is not None: ax.set_xlim(xmin, xmax)
    if ymin is not None or ymax is not None: ax.set_ylim(ymin, ymax)
    if style.get("grid", False): ax.grid(True, alpha=0.3)

    ax.legend(fontsize=style["legend_size"])
    fig.tight_layout()
    
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        fig.savefig(save_path, dpi=style["dpi"], bbox_inches="tight")
        print(f"Saved: {save_path}")
        
    plt.show()

# ============================================================
# 3. DATA LOADING (CSV PARSER)
# ============================================================
def read_samples_to_spectra(filename):
    """Read repeated Name,X,Y blocks from one CSV file and return a list of Spectrum objects."""
    samples = {}
    with open(filename, "r", newline="", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        try:
            header = next(reader)
        except StopIteration:
            raise ValueError(f"Empty file: {filename}")

        if len(header) < 3 or len(header) % 3 != 0:
            raise ValueError(f"{filename}: number of columns ({len(header)}) must be a multiple of 3.")

        n_samples = len(header) // 3

        for row_number, row in enumerate(reader, start=2):
            if not row or all(not cell.strip() for cell in row):
                continue

            if len(row) < 3 * n_samples:
                row = row + [""] * (3 * n_samples - len(row))
            if len(row) > 3 * n_samples:
                raise ValueError(f"{filename}, row {row_number}: found {len(row)} columns; expected {3 * n_samples}.")

            for i in range(n_samples):
                name = row[3*i].strip()
                x_text = row[3*i + 1].strip()
                y_text = row[3*i + 2].strip()

                if not name or not x_text or not y_text:
                    continue

                try:
                    x = float(x_text)
                    y = float(y_text)
                except ValueError:
                    continue

                samples.setdefault(name, ([], []))
                samples[name][0].append(x)
                samples[name][1].append(y)

    # Convert to Spectrum objects
    spectra_list = []
    for name, (x_list, y_list) in samples.items():
        s = Spectrum(N=name, X=np.array(x_list), Y=np.array(y_list))
        spectra_list.append(s)
        
    return spectra_list

def load_all_spectra(folder_path, pattern="*.csv"):
    """Reads all CSVs and dumps EVERY spectrum into a single dictionary."""
    folder = Path(folder_path)
    files = sorted(folder.glob(pattern))
    
    global_spectra_dict = {}
    for file in files:
        try:
            file_spectra = read_samples_to_spectra(file)
            for s in file_spectra:
                # Si hay nombres repetidos en diferentes archivos, puedes usar s.N 
                # o forzar un nombre único agregando el nombre del archivo: f"{file.stem}_{s.N}"
                # Aquí usamos s.N por simplicidad.
                global_spectra_dict[s.N] = s 
        except Exception as e:
            print(f"Skipping {file.name}: {e}")
            
    return global_spectra_dict

# ============================================================
# 4. EXECUTION & SELECTION
# ============================================================
FOLDER_PATH = r"H:\FUBerlin\PhD Update Presentations\AboutDWCNTs_KIT_Shivani\DATA"

# 1. Cargamos TODOS los espectros de la carpeta en un solo diccionario
spectra_dict = load_all_spectra(FOLDER_PATH, "*.csv")
print(f"Archivos procesados. Se encontraron {len(spectra_dict)} espectros únicos:")
for key in spectra_dict.keys():
    print(f" - {key}")

print("\n")

# 2. SELECCIÓN MAESTRA (Igual que en tus otros códigos)
# Aquí eliges qué espectros graficar, cómo llamarlos en la leyenda y qué color usar,
# sin importar de qué archivo CSV provengan originalmente.

# (Clave_en_diccionario, Nombre_para_leyenda, Color, Linestyle)
spectra_config = [
    # Ejemplos (reemplaza los nombres con los que se imprimieron arriba)
    # ("Sample_1_Raw_Name", "Clean Name 1", "#0072B2", "-"),
    # ("Sample_2_Raw_Name", "Clean Name 2", "#D55E00", "--"),
]

# Construir la lista final de objetos a graficar
Selection = []
for key, name, col, ls in spectra_config:
    if key in spectra_dict:
        s = spectra_dict[key].copy() # Importante: hacer copy para no modificar el original
        s.N = name
        s.color = col
        s.linestyle = ls
        Selection.append(s)
    else:
        print(f"Advertencia: El espectro '{key}' no existe en la carpeta.")

# 3. GRAFICAR
if len(Selection) > 0:
    PlotSpectra(
        spectra_list=Selection,
        style=poster_style,
        xlabel="X Axis (Units)",
        ylabel="Y Axis (Units)",
        title="My Custom Combined Plot",
        # xmin=0, xmax=100, # (Opcional) recortes de ejes
    )
else:
    print("No has seleccionado espectros en 'spectra_config' todavía.")

Archivos procesados. Se encontraron 18 espectros únicos:
 - Film T1 S-SWCNTs
 - Film T1 S-SWCNTs Converted
 - Film T2 S-SWCNTs
 - Film T2 S-SWCNTs Converted
 - Film T9 M-SWCNTs
 - Film T9 M-SWCNTs Converted
 - Film T1S (SWCNT)
 - Film T1S (Converted to DWCNT)
 - Film T2S (SWCNT)
 - Film T2S (Converted to DWCNT)
 - Film T9M (SWCNT)
 - Film T9M (Converted to DWCNT)
 - Sol. T1 S-SWCNTs Converted
 - Sol.T4 S-SWCNTs Converted
 - Sol.T4P S-SWCNTs Converted
 - Sol.T1 S-SWCNTs Converted
 - Test Film Sample (SWCNT)
 - Test Film Sample (Converted to DWCNT)


No has seleccionado espectros en 'spectra_config' todavía.
